# Part 4 — Zero-Shot Robust Loss Evaluation (RunPod GPU)

This notebook runs the final Part 4 pipeline for **robust loss evaluation under label noise**.

## Core design
- **No generative VLM dependency in the main experiment**
- Pretrained CLIP-style logits on clean images
- Inject label noise into evaluation labels
- Compare robust losses across batteries A/B/C/D

## Datasets
- PathMNIST
- DermaMNIST

## Models (non-generative)
- `google/medsiglip-448`
- `openai/clip-vit-base-patch32`
- `vinid/plip`
- `microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224`

## RunPod checklist
| Step | Action |
|------|--------|
| 1 | Upload this notebook + `part4_Multimodal_Vision_Robust_Experiments.py` |
| 2 | Set `HF_TOKEN` in RunPod environment (needed for gated MedSigLIP) |
| 3 | Run all cells in order |
| 4 | Download the zip file before stopping pod |


In [ ]:
# ── Cell 1: Install pinned dependencies (RunPod) ─────────────────────────────
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--root-user-action=ignore", *pkgs]
    )

pip_install([
    "typing_extensions>=4.10.0",
    "torch>=2.6.0",
    "torchvision>=0.21.0",
    "transformers>=4.46.0",
    "datasets>=2.20.0",
    "accelerate>=0.33.0",
    "huggingface_hub>=0.24.0",
    "medmnist>=3.0.2",
    "open_clip_torch>=2.24.0",
    "ftfy>=6.2.0",
    "regex>=2024.0.0",
    "scikit-learn>=1.3.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "pandas>=2.1.0",
    "tqdm>=4.66.0",
    "pillow>=10.0.0",
])

print("✓ Install complete.", sys.version)
print("If this is a fresh container install, restart kernel once before running next cells.")


In [ ]:
# ── Cell 2: Runtime sanity checks ─────────────────────────────────────────────
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return "not-installed"

print("typing_extensions:", ver("typing_extensions"))
print("torch:", ver("torch"))
print("transformers:", ver("transformers"))
print("medmnist:", ver("medmnist"))
print("open_clip_torch:", ver("open_clip_torch"))

import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"✓ GPU : {dev.name} | VRAM={dev.total_memory/1024**3:.1f} GB")
    print(f"  CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
else:
    print("⚠ No GPU found. Use a GPU pod for practical runtime.")


In [ ]:
# ── Cell 3: Configure workspace & cache paths ──────────────────────────────────
import os
from pathlib import Path

# ── Output directory ──────────────────────────────────────────────────────────
WS = Path(os.environ.get("ROBUST_NN_WORKSPACE", "/workspace/runpod_outputs"))
WS.mkdir(parents=True, exist_ok=True)
os.environ["ROBUST_NN_WORKSPACE"] = str(WS)

# ── Cache directories to a large persistent volume (if present) ───────────────
HF_HOME   = "/workspace/.cache/hf"
TORCH_HOME = "/workspace/.cache/torch"
for d in [HF_HOME, TORCH_HOME, f"{HF_HOME}/datasets"]:
    Path(d).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME",           HF_HOME)
os.environ.setdefault("TRANSFORMERS_CACHE", HF_HOME + "/models")
os.environ.setdefault("TORCH_HOME",         TORCH_HOME)

# ── HF token (read-only check — never print the token value itself) ────────────
hf_token = os.environ.get("HF_TOKEN", "")
print(f"✓ ROBUST_NN_WORKSPACE : {WS}")
print(f"  HF_TOKEN set        : {bool(hf_token)}  (length {len(hf_token)})")
print(f"  HF_HOME             : {HF_HOME}")
print(f"  TORCH_HOME          : {TORCH_HOME}")


In [ ]:
# ── Cell 4: Locate part4 Python script ────────────────────────────────────────
from pathlib import Path

SCRIPT_NAME = "part4_Multimodal_Vision_Robust_Experiments.py"
SEARCH_ROOTS = [
    "/workspace",
    "/workspace/robustNN",
    "/workspace/Robust-NN-learning",
    "/workspace/robust-nn-learning",
    "/notebooks",
    ".",
]

SCRIPT_PATH = None
for root in SEARCH_ROOTS:
    p = Path(root) / SCRIPT_NAME
    if p.exists():
        SCRIPT_PATH = str(p)
        break
    # also walk one level deep
    for child in Path(root).glob(f"*/{SCRIPT_NAME}"):
        SCRIPT_PATH = str(child)
        break
    if SCRIPT_PATH:
        break

if SCRIPT_PATH:
    print(f"✓ Found: {SCRIPT_PATH}")
else:
    raise FileNotFoundError(
        f"Cannot find {SCRIPT_NAME!r}.\n"
        f"Searched: {SEARCH_ROOTS}\n"
        "Please upload the file to one of those directories."
    )


In [ ]:
# ── Cell 5: Run Part 4 robust-loss experiment pipeline ───────────────────────
# Batteries:
#   A: clean labels
#   B: uniform label noise
#   C: class-dependent label noise
#   D: GCE q-parameter sweep
#
# Optional env controls before running this cell:
#   ROBUST_NN_MODEL_ONLY=MedSigLIP|CLIP|PLIP|BiomedCLIP
#   ROBUST_NN_MAX_SAMPLES=1200
#   ROBUST_NN_QUICK_RUN=1

assert SCRIPT_PATH, "SCRIPT_PATH not set — run Cell 4 first."
import runpy
runpy.run_path(SCRIPT_PATH, run_name="__main__")


## After the run — inspect and archive results

Run the next three cells (6, 7, 8) in order to:
1. **List** every output file with its size
2. **Preview** the accuracy summary table
3. **Zip** the entire output directory for download


In [ ]:
# ── Cell 6: List saved output files ────────────────────────────────────────────
import os
from pathlib import Path

ws = Path(os.environ["ROBUST_NN_WORKSPACE"])
files = sorted(ws.rglob("*"))
print(f"Output folder  : {ws}\n")
fmt = "{:<7s}  {}"
print(fmt.format("SIZE", "PATH"))
print("-" * 70)
for f in files:
    if f.is_file():
        sz = f.stat().st_size
        if sz >= 1024**2:
            label = f"{sz/1024**2:6.1f}M"
        elif sz >= 1024:
            label = f"{sz/1024:6.1f}K"
        else:
            label = f"{sz:6d}B"
        print(fmt.format(label, f.relative_to(ws)))


In [ ]:
# ── Cell 7: Preview summary table ─────────────────────────────────────────────
import pandas as pd, os
from pathlib import Path

summary_path = (
    Path(os.environ["ROBUST_NN_WORKSPACE"])
    / os.environ.get("ROBUST_NN_RESULTS_SUBDIR", "results_multimodal_vision")
    / "summary_all.csv"
)
if summary_path.exists():
    df = pd.read_csv(summary_path)
    print(f"Summary table  ({len(df)} rows × {len(df.columns)} cols)\n")
    cols = [
        "dataset", "model_key", "battery", "noise_type", "noise_rate",
        "loss", "loss_value", "acc_clean_labels", "acc_noisy_labels"
    ]
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))
else:
    print("summary_all.csv not yet generated — run Cell 5 first.")


In [ ]:
# ── Cell 8 (FINAL): Zip entire workspace for download ─────────────────────────
# Run this BEFORE shutting down the pod — RunPod deletes disks on pod termination.
import shutil, os
from pathlib import Path

ws  = Path(os.environ["ROBUST_NN_WORKSPACE"])
out = ws.parent / (ws.name + "_part4_archive")
arc = shutil.make_archive(str(out), "zip", root_dir=ws.parent, base_dir=ws.name)
size_mb = Path(arc).stat().st_size / 1024**2
print(f"✓ Archive created : {arc}")
print(f"  Size            : {size_mb:.1f} MB")
print("  → Download this file from the RunPod file browser before stopping the pod.")
